# Comparison of Hyperparameter Optimization Methods and SHAP Interpretation of LightGBM for Medication Adherence Prediction in Non-Communicable Disease Patients

**Ihya' Nashirudin Abrar** — Master of Informatics, Universitas Ahmad Dahlan, Indonesia (2026)
*Original thesis title (Indonesian): Perbandingan Metode Optimasi Hiperparameter dan Interpretasi SHAP pada LightGBM untuk Prediksi Kepatuhan Pengobatan Pasien Penyakit Tidak Menular*

This notebook contains **the complete thesis experiment, end to end**, as one reproducible pipeline:

| Part | Content | Thesis section |
|---|---|---|
| 1 | Setup, libraries, random seed | 3.2.1 |
| 2 | Data loading and exploration | 3.2.2, 4.1 |
| 3 | Pre-processing (deduplication, encoding, feature engineering, stratified split) | 3.3.2, 4.1 |
| 4 | Preliminary study: five boosting algorithms (rationale for choosing LightGBM) | Appendix 2 |
| 5 | **Stage 1** — Hyperparameter optimization (6 methods) + Friedman and Nemenyi tests | 3.3.3, 4.2 |
| 6 | Final model training | 4.2 |
| 7 | **Stage 2** — Four-level SHAP interpretation | 3.3.4, 4.3 |
| 8 | Final model evaluation (per class, 95% bootstrap CI, exploratory Youden threshold) | 3.3.5, 4.4 |
| 9 | Sensitivity analysis (class weighting, claim-unit feature ablation, baseline vs final) | 4.5 |
| 10 | Summary | Chapter 5 |

All stochastic components use `random_state = 42`. A full run takes roughly 35–45 minutes on a CPU, mostly in Stage 1.

## 1. Setup and Libraries

In [1]:
import warnings, time, json, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, GridSearchCV,
                                      RandomizedSearchCV)
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, roc_curve, auc,
                             classification_report)
from scipy import stats
from scipy.stats import randint, uniform, loguniform

from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import BayesSearchCV
from skopt.space import Integer, Real
import scikit_posthocs as sp

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'Final Prepared Dataset - Diabetes and Hypertension Data.xlsx'
FIG = str(ROOT / 'figures')
OUT = str(ROOT / 'outputs')
os.makedirs(FIG, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# consistent visual theme
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'figure.facecolor': 'white', 'savefig.bbox': 'tight',
})
PALETTE = ['#2E5A87', '#C0504D', '#4F8A5B', '#E39A2C', '#7B5EA7']
import sys, sklearn, lightgbm, xgboost, catboost, scipy
print('Libraries loaded. Key versions:')
print(f'  python={sys.version.split()[0]} sklearn={sklearn.__version__} lightgbm={lightgbm.__version__} '
      f'optuna={optuna.__version__} shap={shap.__version__} scipy={scipy.__version__}')
print(f'  xgboost={xgboost.__version__} catboost={catboost.__version__}')

Libraries loaded. Key versions:
  python=3.11.1 sklearn=1.8.0 lightgbm=4.6.0 optuna=4.8.0 shap=0.49.1 scipy=1.15.3
  xgboost=1.6.2 catboost=1.2.10


## 2. Data Loading and Exploration

Public dataset *Final Prepared Dataset — Diabetes and Hypertension Data* (Kanyongo, 2024; Kanyongo et al., 2025b), CC0 1.0 licence, DOI [10.17632/zkp7sbbx64.2](https://doi.org/10.17632/zkp7sbbx64.2). The target **ADHERENCE** is defined from the number of monthly medication refills in 12 months: **ADHERENT** = 9–12 refills (≥ 75%), **NON-ADHERENT** = 1–8 refills.

In [2]:
df = pd.read_excel(DATA_PATH)
print('Raw data shape :', df.shape)
print('Columns        :', list(df.columns))
print('\nTarget distribution (ADHERENCE):')
print(df['ADHERENCE'].value_counts())
print('\nProportion (%):')
print((df['ADHERENCE'].value_counts(normalize=True) * 100).round(2))
df.head()

Raw data shape : (24084, 12)
Columns        : ['ADHERENCE', 'AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION']

Target distribution (ADHERENCE):
ADHERENCE
NON-ADHERENT    14415
ADHERENT         9669
Name: count, dtype: int64

Proportion (%):
ADHERENCE
NON-ADHERENT    59.85
ADHERENT        40.15
Name: proportion, dtype: float64


,ADHERENCE,AGE,ANNUALCONTRIBUTION,ANNUALCLAIMAMOUNT,UNITSTOTAL,GENDER_M,SCHEMETYPE_MEDIUM,SCHEMETYPE_PREMIUM,DIAGNOSIS_HYPERTENSION,COVERTYPE_STANDARD,COMORBIDITY_NO_COMORBIDITY,COMPLICATIONDEVELOPMENT_NO_COMPLICATION
0,NON-ADHERENT,24,690384.0,10400.00,60.0,False,False,False,False,True,True,True
1,NON-ADHERENT,53,2718696.0,306366.53,1380.0,True,True,False,False,True,True,True
2,NON-ADHERENT,46,808284.0,77480.00,600.0,True,False,False,False,True,False,True
3,NON-ADHERENT,76,808284.0,198800.58,840.0,True,False,False,False,True,False,True
4,ADHERENT,69,808284.0,116872.04,720.0,False,False,False,False,True,False,True


In [3]:
# --- Figure: target distribution ---
counts = df['ADHERENCE'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
bars = ax[0].bar(counts.index, counts.values, color=[PALETTE[1], PALETTE[0]], edgecolor='black', width=0.6)
ax[0].set_title('Target Class Distribution')
ax[0].set_ylabel('Number of Patients')
for b, v in zip(bars, counts.values):
    ax[0].text(b.get_x()+b.get_width()/2, v+150, f'{v:,}\n({v/counts.sum()*100:.1f}%)',
               ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].set_ylim(0, counts.max()*1.18)
ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90,
          colors=[PALETTE[1], PALETTE[0]], wedgeprops=dict(edgecolor='white', linewidth=2),
          textprops=dict(fontsize=11))
ax[1].set_title('Target Class Proportion')
plt.tight_layout(); plt.savefig(f'{FIG}/fig01_target_distribution.png'); plt.show()
imbalance = counts.max()/counts.min()
print(f'Imbalance ratio = {imbalance:.2f} : 1')

Imbalance ratio = 1.49 : 1


## 3. Data Pre-processing

1. Remove exact duplicate rows.
2. Encode the target as `ADHERENT = 1`, `NON-ADHERENT = 0`; convert boolean features to integer 1/0.
3. Row-wise feature engineering (no aggregate statistics, so it is safe to do before the split): `LOG_CLAIM`, `LOG_UNITS`, `LOG_CONTRIB` (log1p), `CLAIM_PER_UNIT`, `CLAIM_RATIO` (ratio + log1p) and `UNITS_PER_AGE` → **17 features**.
4. Stratified 80:20 train/test split.

No scaling (LightGBM is tree-based), no class weighting and no SMOTE in the main model. The effect of class weighting is tested separately in Part 9.

In [4]:
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicate rows removed : {n_dup}  ->  {df.shape[0]} unique records remain')

df_p = df.copy()
df_p['ADHERENCE'] = (df_p['ADHERENCE'] == 'ADHERENT').astype(int)
for c in df_p.columns:
    if df_p[c].dtype == 'bool':
        df_p[c] = df_p[c].astype(int)

y = df_p['ADHERENCE']
X = df_p.drop(columns=['ADHERENCE']).copy()

# --- feature engineering (row-wise, no aggregate statistics) ---
X['LOG_CLAIM']      = np.log1p(X['ANNUALCLAIMAMOUNT'])
X['LOG_UNITS']      = np.log1p(X['UNITSTOTAL'])
X['LOG_CONTRIB']    = np.log1p(X['ANNUALCONTRIBUTION'])
X['CLAIM_PER_UNIT'] = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['UNITSTOTAL'] + 1))
X['CLAIM_RATIO']    = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['ANNUALCONTRIBUTION'] + 1))
X['UNITS_PER_AGE']  = X['UNITSTOTAL'] / (X['AGE'] + 1)

print(f'Number of features after engineering : {X.shape[1]}')
print('Features:', list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f'\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print('Train class proportion:', dict(y_train.value_counts(normalize=True).round(3)))

Duplicate rows removed : 13  ->  24071 unique records remain
Number of features after engineering : 17
Features: ['AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION', 'LOG_CLAIM', 'LOG_UNITS', 'LOG_CONTRIB', 'CLAIM_PER_UNIT', 'CLAIM_RATIO', 'UNITS_PER_AGE']

Train: 19256 samples | Test: 4815 samples
Train class proportion: {0: 0.598, 1: 0.402}


In [5]:
# --- Figure: feature correlation ---
plt.figure(figsize=(11, 9))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.7, 'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix (17 Features)')
plt.tight_layout(); plt.savefig(f'{FIG}/fig02_feature_correlation.png'); plt.show()

## 4. Preliminary Study — Rationale for Choosing LightGBM

Five boosting algorithms are compared briefly with **10-fold stratified CV** to check that LightGBM is among the best performers before it becomes the focus of the thesis.

> This part is **not** the main analysis of the thesis; it only motivates the choice of LightGBM and is reported in a separate publication. Unlike the main model, this preliminary study uses each algorithm's built-in class weighting.

In [6]:
CV10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# balanced sample weights for AdaBoost & GradientBoosting
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()  # scale_pos_weight for XGBoost

def make_models():
    return {
        'XGBoost': XGBClassifier(
            n_estimators=100, learning_rate=0.1, scale_pos_weight=pos_weight,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss',
            use_label_encoder=False, verbosity=0),
        'AdaBoost': AdaBoostClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
        'LightGBM': LGBMClassifier(
            n_estimators=100, learning_rate=0.1, class_weight='balanced',
            random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        'CatBoost': CatBoostClassifier(
            n_estimators=100, learning_rate=0.1, auto_class_weights='Balanced',
            random_state=RANDOM_STATE, verbose=0),
    }

NEEDS_SW = {'AdaBoost', 'Gradient Boosting'}  # use balanced sample_weight

def cv_boosting(name, model):
    rows = []
    for k, (tr, te) in enumerate(CV10.split(X_train, y_train), 1):
        Xtr, Xte = X_train.iloc[tr], X_train.iloc[te]
        ytr, yte = y_train.iloc[tr], y_train.iloc[te]
        mdl = make_models()[name]
        if name in NEEDS_SW:
            sw = compute_sample_weight('balanced', ytr)
            mdl.fit(Xtr, ytr, sample_weight=sw)
        else:
            mdl.fit(Xtr, ytr)
        yp = mdl.predict(Xte); pp = mdl.predict_proba(Xte)[:, 1]
        rows.append(dict(fold=k,
            accuracy=accuracy_score(yte, yp),
            precision=precision_score(yte, yp, average='weighted'),
            recall=recall_score(yte, yp, average='weighted'),
            f1=f1_score(yte, yp, average='weighted'),
            mcc=matthews_corrcoef(yte, yp),
            auc=roc_auc_score(yte, pp)))
    return pd.DataFrame(rows)

boost_folds = {}
print('Running 10-fold CV for 5 boosting algorithms...\n')
for name, mdl in make_models().items():
    t0 = time.time()
    fdf = cv_boosting(name, mdl)
    boost_folds[name] = fdf
    print(f'  {name:20s} acc={fdf.accuracy.mean():.4f}±{fdf.accuracy.std():.4f} '
          f'auc={fdf.auc.mean():.4f}  mcc={fdf.mcc.mean():.4f}  ({time.time()-t0:.1f}s)')

Running 10-fold CV for 5 boosting algorithms...



  XGBoost              acc=0.8184±0.0076 auc=0.9029  mcc=0.6551  (38.1s)


  AdaBoost             acc=0.7873±0.0075 auc=0.8725  mcc=0.6222  (35.9s)


  Gradient Boosting    acc=0.8116±0.0068 auc=0.8992  mcc=0.6492  (87.0s)


  LightGBM             acc=0.8173±0.0088 auc=0.9028  mcc=0.6508  (3.1s)


  CatBoost             acc=0.8123±0.0064 auc=0.9005  mcc=0.6492  (8.1s)


In [7]:
# --- Preliminary study: metric summary ---
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'mcc', 'auc']
boost_summary = pd.DataFrame({
    name: {mk: fdf[mk].mean() for mk in metric_keys} for name, fdf in boost_folds.items()
}).T.round(4)
boost_summary_std = pd.DataFrame({
    name: {mk: fdf[mk].std() for mk in metric_keys} for name, fdf in boost_folds.items()
}).T.round(4)
print('Mean 10-fold CV metrics (preliminary study):')
print(boost_summary.to_string())
best_boost = boost_summary['accuracy'].idxmax()
print(f'\n>> Highest mean CV accuracy: {best_boost}')
boost_summary.to_csv(f'{OUT}/preliminary_boosting_summary.csv')

Mean 10-fold CV metrics (preliminary study):
                   accuracy  precision  recall      f1     mcc     auc
XGBoost              0.8184     0.8423  0.8184  0.8201  0.6551  0.9029
AdaBoost             0.7873     0.8366  0.7873  0.7882  0.6222  0.8725
Gradient Boosting    0.8116     0.8420  0.8116  0.8133  0.6492  0.8992
LightGBM             0.8173     0.8395  0.8173  0.8191  0.6508  0.9028
CatBoost             0.8123     0.8416  0.8123  0.8139  0.6492  0.9005

>> Highest mean CV accuracy: XGBoost


In [8]:
# --- Figure: overall metrics of 5 algorithms ---
fig, ax = plt.subplots(figsize=(12, 5.5))
algos = list(boost_summary.index)
xpos = np.arange(len(metric_keys)); w = 0.15
for i, algo in enumerate(algos):
    vals = boost_summary.loc[algo, metric_keys].values
    errs = boost_summary_std.loc[algo, metric_keys].values
    ax.bar(xpos + i*w, vals, w, yerr=errs, capsize=2.5, label=algo,
           color=PALETTE[i], edgecolor='black', linewidth=0.5)
ax.set_xticks(xpos + w*2); ax.set_xticklabels([m.upper() for m in metric_keys])
ax.set_ylabel('Score'); ax.set_ylim(0.5, 1.0)
ax.set_title('10-Fold CV Metrics — 5 Boosting Algorithms')
ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.08), frameon=False)
plt.tight_layout(); plt.savefig(f'{FIG}/fig03_boosting_metrics.png'); plt.show()

In [9]:
# --- Figure: per-fold accuracy boxplot ---
fig, ax = plt.subplots(figsize=(9.5, 5.2))
box_data = [boost_folds[a]['accuracy'].values for a in algos]
bp = ax.boxplot(box_data, labels=algos, patch_artist=True, widths=0.55,
                medianprops=dict(color='black', linewidth=1.5))
for patch, c in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_ylabel('Accuracy'); ax.set_title('10-Fold CV Accuracy per Algorithm')
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(f'{FIG}/fig04_boosting_boxplot.png'); plt.show()

In [10]:
# --- Holdout fit: ROC + confusion matrix for all algorithms ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
roc_ax = axes[0, 0]
boost_holdout = {}
for i, (name, _) in enumerate(make_models().items()):
    mdl = make_models()[name]
    if name in NEEDS_SW:
        mdl.fit(X_train, y_train, sample_weight=sw_train)
    else:
        mdl.fit(X_train, y_train)
    yp = mdl.predict(X_test); pp = mdl.predict_proba(X_test)[:, 1]
    boost_holdout[name] = dict(
        accuracy=accuracy_score(y_test, yp), auc=roc_auc_score(y_test, pp),
        mcc=matthews_corrcoef(y_test, yp), model=mdl)
    fpr, tpr, _ = roc_curve(y_test, pp)
    roc_ax.plot(fpr, tpr, color=PALETTE[i], lw=2,
                label=f'{name} (AUC={auc(fpr,tpr):.3f})')
roc_ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.6)
roc_ax.set_xlabel('False Positive Rate'); roc_ax.set_ylabel('True Positive Rate')
roc_ax.set_title('ROC Curves (Holdout)'); roc_ax.legend(fontsize=8, loc='lower right')

for i, name in enumerate(make_models().keys()):
    r, c = divmod(i+1, 3)
    cm = confusion_matrix(y_test, boost_holdout[name]['model'].predict(X_test))
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=axes[r, c],
                xticklabels=['NON-ADH','ADH'], yticklabels=['NON-ADH','ADH'])
    axes[r, c].set_title(f'{name}\nacc={boost_holdout[name]["accuracy"]:.3f}', fontsize=10)
    axes[r, c].set_xlabel('Predicted'); axes[r, c].set_ylabel('Actual')
plt.tight_layout(); plt.savefig(f'{FIG}/fig05_boosting_roc_cm.png'); plt.show()

In [11]:
# --- Wilcoxon signed-rank test between algorithms (per-fold accuracy) ---
pairs = algos
wilcox_p = pd.DataFrame(np.ones((len(pairs), len(pairs))), index=pairs, columns=pairs)
for i in range(len(pairs)):
    for j in range(len(pairs)):
        if i != j:
            a = boost_folds[pairs[i]]['accuracy'].values
            b = boost_folds[pairs[j]]['accuracy'].values
            try:
                stat, p = stats.wilcoxon(a, b)
            except ValueError:
                p = 1.0
            wilcox_p.iloc[i, j] = p
plt.figure(figsize=(7.5, 6))
sns.heatmap(wilcox_p, annot=True, fmt='.3f', cmap='RdYlGn_r', center=0.05,
            vmin=0, vmax=0.2, linewidths=0.5, square=True,
            cbar_kws={'label': 'p-value'})
plt.title("Wilcoxon Signed-Rank Test — Pairwise p-values\n(p < 0.05 = significant difference)")
plt.tight_layout(); plt.savefig(f'{FIG}/fig06_wilcoxon.png'); plt.show()
print('Wilcoxon p-value (row vs column):')
print(wilcox_p.round(4).to_string())

Wilcoxon p-value (row vs column):
                   XGBoost  AdaBoost  Gradient Boosting  LightGBM  CatBoost
XGBoost             1.0000     0.002             0.0039    0.5410    0.0020
AdaBoost            0.0020     1.000             0.0020    0.0020    0.0020
Gradient Boosting   0.0039     0.002             1.0000    0.0254    0.5391
LightGBM            0.5410     0.002             0.0254    1.0000    0.0098
CatBoost            0.0020     0.002             0.5391    0.0098    1.0000


## 5. Stage 1 — LightGBM Hyperparameter Optimization

Six HPO methods from four paradigms: exhaustive (Grid Search), stochastic (Random Search), multi-fidelity (Halving Random Search) and Bayesian (Optuna-TPE, Hyperopt-TPE, scikit-optimize-GP). Objective: *accuracy* in an **inner 5-fold CV** on the training set. The best configuration of each method is then re-evaluated with **10-fold stratified CV** (identical splits and seed).

Design notes (see also thesis sections 3.3.3 and 4.6):
- Random Search and the three Bayesian methods search the same 9-hyperparameter space with a budget of **40 trials**; Halving uses the same distributions and ends up with **1,444 evaluations**; Grid Search covers only **5 hyperparameters (162 combinations)**.
- `subsample` is **inactive** because `subsample_freq = 0` (LightGBM default), so its value is not interpreted.
- Parallelism is not uniform: Grid, Random and Halving parallelise over candidates × folds (`n_jobs=-1`), whereas Optuna, Hyperopt and scikit-optimize evaluate candidates sequentially. Run times are therefore only indicative.
- This is not full nested cross-validation; the Friedman–Nemenyi tests are treated as an internal analysis.

In [12]:
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
DEFAULT = {'random_state': RANDOM_STATE, 'verbose': -1, 'n_jobs': 1}
N_TRIALS = 40

def run_10fold(params):
    rows = []
    for k, (tr, te) in enumerate(CV10.split(X_train, y_train), 1):
        Xtr, Xte = X_train.iloc[tr], X_train.iloc[te]
        ytr, yte = y_train.iloc[tr], y_train.iloc[te]
        m = LGBMClassifier(**{**params, **DEFAULT})
        m.fit(Xtr, ytr)
        yp = m.predict(Xte); pp = m.predict_proba(Xte)[:, 1]
        rows.append(dict(fold=k,
            accuracy=accuracy_score(yte, yp),
            precision=precision_score(yte, yp, average='weighted'),
            recall=recall_score(yte, yp, average='weighted'),
            f1=f1_score(yte, yp, average='weighted'),
            mcc=matthews_corrcoef(yte, yp),
            auc=roc_auc_score(yte, pp)))
    return pd.DataFrame(rows)

hpo_folds = {}; hpo_params = {}; hpo_time = {}; hpo_history = {}

In [13]:
# --- Baseline (no tuning) ---
t0 = time.time()
hpo_params['Baseline'] = {}
hpo_folds['Baseline'] = run_10fold({})
hpo_time['Baseline'] = 0.0
print(f"Baseline           acc={hpo_folds['Baseline'].accuracy.mean():.4f}")

Baseline           acc=0.8218


In [14]:
# --- Grid Search (162 combinations) ---
grid_space = {'n_estimators':[100,300,500], 'learning_rate':[0.01,0.05,0.1],
              'max_depth':[-1,6,10], 'num_leaves':[31,63,127], 'min_child_samples':[20,50]}
t0 = time.time()
gs = GridSearchCV(LGBMClassifier(**DEFAULT), grid_space, cv=CV5, scoring='accuracy',
                  n_jobs=-1, refit=False)
gs.fit(X_train, y_train)
hpo_time['Grid Search'] = time.time()-t0
hpo_params['Grid Search'] = gs.best_params_
hpo_history['Grid Search'] = np.maximum.accumulate(gs.cv_results_['mean_test_score'])
hpo_folds['Grid Search'] = run_10fold(gs.best_params_)
print(f"Grid Search  ({len(gs.cv_results_['params'])} combinations, {hpo_time['Grid Search']:.0f}s)  "
      f"best_cv={gs.best_score_:.4f}  cv10={hpo_folds['Grid Search'].accuracy.mean():.4f}")

Grid Search  (162 combinations, 632s)  best_cv=0.8232  cv10=0.8211


In [15]:
# --- Random Search ---
random_space = {'n_estimators':randint(100,1000), 'learning_rate':loguniform(0.005,0.2),
    'max_depth':randint(3,13), 'num_leaves':randint(15,255), 'min_child_samples':randint(5,100),
    'subsample':uniform(0.5,0.5), 'colsample_bytree':uniform(0.5,0.5),
    'reg_alpha':loguniform(1e-8,10), 'reg_lambda':loguniform(1e-8,10)}
t0 = time.time()
rs = RandomizedSearchCV(LGBMClassifier(**DEFAULT), random_space, n_iter=N_TRIALS, cv=CV5,
                        scoring='accuracy', n_jobs=-1, random_state=RANDOM_STATE, refit=False)
rs.fit(X_train, y_train)
hpo_time['Random Search'] = time.time()-t0
hpo_params['Random Search'] = dict(rs.best_params_)
hpo_history['Random Search'] = np.maximum.accumulate(rs.cv_results_['mean_test_score'])
hpo_folds['Random Search'] = run_10fold(rs.best_params_)
print(f"Random Search ({N_TRIALS} trials, {hpo_time['Random Search']:.0f}s)  "
      f"best_cv={rs.best_score_:.4f}  cv10={hpo_folds['Random Search'].accuracy.mean():.4f}")

Random Search (40 trials, 185s)  best_cv=0.8235  cv10=0.8241


In [16]:
# --- Halving Random Search ---
t0 = time.time()
hrs = HalvingRandomSearchCV(LGBMClassifier(**DEFAULT), random_space, cv=CV5,
    scoring='accuracy', n_jobs=-1, random_state=RANDOM_STATE, factor=3,
    resource='n_samples', refit=False)
hrs.fit(X_train, y_train)
hpo_time['Halving'] = time.time()-t0
hpo_params['Halving'] = dict(hrs.best_params_)
hpo_history['Halving'] = np.maximum.accumulate(hrs.cv_results_['mean_test_score'])
hpo_folds['Halving'] = run_10fold(hrs.best_params_)
print(f"Halving ({len(hrs.cv_results_['params'])} evals, {hpo_time['Halving']:.0f}s)  "
      f"best_cv={hrs.best_score_:.4f}  cv10={hpo_folds['Halving'].accuracy.mean():.4f}")

Halving (1444 evals, 285s)  best_cv=0.8187  cv10=0.8223


In [17]:
# --- Optuna TPE ---
def optuna_obj(trial):
    params = {'n_estimators':trial.suggest_int('n_estimators',100,1000),
        'learning_rate':trial.suggest_float('learning_rate',0.005,0.2,log=True),
        'max_depth':trial.suggest_int('max_depth',3,12),
        'num_leaves':trial.suggest_int('num_leaves',15,255),
        'min_child_samples':trial.suggest_int('min_child_samples',5,100),
        'subsample':trial.suggest_float('subsample',0.5,1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.5,1.0),
        'reg_alpha':trial.suggest_float('reg_alpha',1e-8,10.0,log=True),
        'reg_lambda':trial.suggest_float('reg_lambda',1e-8,10.0,log=True), **DEFAULT}
    return cross_val_score(LGBMClassifier(**params), X_train, y_train,
                           cv=CV5, scoring='accuracy', n_jobs=-1).mean()
t0 = time.time()
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study.optimize(optuna_obj, n_trials=N_TRIALS, show_progress_bar=False)
hpo_time['Optuna TPE'] = time.time()-t0
hpo_params['Optuna TPE'] = study.best_params
hpo_history['Optuna TPE'] = np.maximum.accumulate([t.value for t in study.trials if t.value is not None])
hpo_folds['Optuna TPE'] = run_10fold(study.best_params)
print(f"Optuna TPE ({N_TRIALS} trials, {hpo_time['Optuna TPE']:.0f}s)  "
      f"best_cv={study.best_value:.4f}  cv10={hpo_folds['Optuna TPE'].accuracy.mean():.4f}")

Optuna TPE (40 trials, 455s)  best_cv=0.8235  cv10=0.8234


In [18]:
# --- Hyperopt TPE ---
hp_space = {'n_estimators':hp.quniform('n_estimators',100,1000,1),
    'learning_rate':hp.loguniform('learning_rate',np.log(0.005),np.log(0.2)),
    'max_depth':hp.quniform('max_depth',3,12,1), 'num_leaves':hp.quniform('num_leaves',15,255,1),
    'min_child_samples':hp.quniform('min_child_samples',5,100,1),
    'subsample':hp.uniform('subsample',0.5,1.0), 'colsample_bytree':hp.uniform('colsample_bytree',0.5,1.0),
    'reg_alpha':hp.loguniform('reg_alpha',np.log(1e-8),np.log(10.0)),
    'reg_lambda':hp.loguniform('reg_lambda',np.log(1e-8),np.log(10.0))}
hp_hist = []
def hp_obj(p):
    params = {'n_estimators':int(p['n_estimators']),'learning_rate':p['learning_rate'],
        'max_depth':int(p['max_depth']),'num_leaves':int(p['num_leaves']),
        'min_child_samples':int(p['min_child_samples']),'subsample':p['subsample'],
        'colsample_bytree':p['colsample_bytree'],'reg_alpha':p['reg_alpha'],
        'reg_lambda':p['reg_lambda'], **DEFAULT}
    s = cross_val_score(LGBMClassifier(**params), X_train, y_train, cv=CV5,
                        scoring='accuracy', n_jobs=-1).mean()
    hp_hist.append(s); return {'loss':-s, 'status':STATUS_OK}
t0 = time.time()
trials_hp = Trials()
best_raw = fmin(hp_obj, hp_space, algo=tpe.suggest, max_evals=N_TRIALS,
                trials=trials_hp, rstate=np.random.default_rng(RANDOM_STATE), verbose=False)
hpo_time['Hyperopt TPE'] = time.time()-t0
hp_best = {'n_estimators':int(best_raw['n_estimators']),'learning_rate':best_raw['learning_rate'],
    'max_depth':int(best_raw['max_depth']),'num_leaves':int(best_raw['num_leaves']),
    'min_child_samples':int(best_raw['min_child_samples']),'subsample':best_raw['subsample'],
    'colsample_bytree':best_raw['colsample_bytree'],'reg_alpha':best_raw['reg_alpha'],
    'reg_lambda':best_raw['reg_lambda']}
hpo_params['Hyperopt TPE'] = hp_best
hpo_history['Hyperopt TPE'] = np.maximum.accumulate(hp_hist)
hpo_folds['Hyperopt TPE'] = run_10fold(hp_best)
print(f"Hyperopt TPE ({N_TRIALS} trials, {hpo_time['Hyperopt TPE']:.0f}s)  "
      f"cv10={hpo_folds['Hyperopt TPE'].accuracy.mean():.4f}")

Hyperopt TPE (40 trials, 747s)  cv10=0.8226


In [19]:
# --- scikit-optimize GP ---
skopt_space = {'n_estimators':Integer(100,1000),'learning_rate':Real(0.005,0.2,prior='log-uniform'),
    'max_depth':Integer(3,12),'num_leaves':Integer(15,255),'min_child_samples':Integer(5,100),
    'subsample':Real(0.5,1.0),'colsample_bytree':Real(0.5,1.0),
    'reg_alpha':Real(1e-8,10.0,prior='log-uniform'),'reg_lambda':Real(1e-8,10.0,prior='log-uniform')}
t0 = time.time()
bs = BayesSearchCV(LGBMClassifier(**DEFAULT), skopt_space, n_iter=N_TRIALS, cv=CV5,
                   scoring='accuracy', n_jobs=-1, random_state=RANDOM_STATE, refit=False)
bs.fit(X_train, y_train)
hpo_time['scikit-optimize GP'] = time.time()-t0
hpo_params['scikit-optimize GP'] = dict(bs.best_params_)
hpo_history['scikit-optimize GP'] = np.maximum.accumulate(bs.cv_results_['mean_test_score'])
hpo_folds['scikit-optimize GP'] = run_10fold(dict(bs.best_params_))
print(f"scikit-optimize GP ({N_TRIALS} trials, {hpo_time['scikit-optimize GP']:.0f}s)  "
      f"best_cv={bs.best_score_:.4f}  cv10={hpo_folds['scikit-optimize GP'].accuracy.mean():.4f}")

scikit-optimize GP (40 trials, 4004s)  best_cv=0.8232  cv10=0.8234


In [20]:
# --- Stage 1 (HPO) summary ---
ORDER = ['Baseline','Grid Search','Random Search','Halving','Optuna TPE','Hyperopt TPE','scikit-optimize GP']
hpo_summary = pd.DataFrame({
    name: {mk: hpo_folds[name][mk].mean() for mk in metric_keys} for name in ORDER
}).T
hpo_summary['acc_std'] = [hpo_folds[n]['accuracy'].std() for n in ORDER]
hpo_summary['time_s'] = [hpo_time[n] for n in ORDER]
hpo_summary = hpo_summary.round(4)
print('10-fold CV summary — 7 configurations (Stage 1):')
print(hpo_summary.to_string())
best_hpo = hpo_summary.loc[ORDER, 'accuracy'].idxmax()
print(f'\n>> Highest mean CV accuracy: {best_hpo}')
print(f'   Highest MCC: {hpo_summary["mcc"].idxmax()}')
hpo_summary.to_csv(f'{OUT}/stage1_hpo_summary.csv')
json.dump({k: {kk: (float(vv) if isinstance(vv,(int,float,np.floating)) else vv)
               for kk,vv in v.items()} for k,v in hpo_params.items()},
          open(f'{OUT}/stage1_best_params.json','w'), indent=2)

10-fold CV summary — 7 configurations (Stage 1):
                    accuracy  precision  recall      f1     mcc     auc  acc_std     time_s
Baseline              0.8218     0.8307  0.8218  0.8233  0.6430  0.9025   0.0079     0.0000
Grid Search           0.8211     0.8306  0.8211  0.8226  0.6423  0.9033   0.0062   632.3898
Random Search         0.8241     0.8330  0.8241  0.8255  0.6477  0.9046   0.0090   185.2108
Halving               0.8223     0.8307  0.8223  0.8238  0.6434  0.9026   0.0074   284.9333
Optuna TPE            0.8234     0.8325  0.8234  0.8248  0.6464  0.9041   0.0070   455.1507
Hyperopt TPE          0.8226     0.8322  0.8226  0.8241  0.6455  0.9035   0.0074   747.4926
scikit-optimize GP    0.8234     0.8331  0.8234  0.8249  0.6473  0.9046   0.0074  4003.8932

>> Highest mean CV accuracy: Random Search
   Highest MCC: Random Search


In [21]:
# --- Figure: CV accuracy of 7 configurations ---
fig, ax = plt.subplots(figsize=(11, 5.5))
accs = hpo_summary.loc[ORDER, 'accuracy'].values
stds = hpo_summary.loc[ORDER, 'acc_std'].values
colors = ['#999999'] + [PALETTE[0]]*3 + [PALETTE[2]]*3
bars = ax.bar(range(len(ORDER)), accs, yerr=stds, capsize=4, color=colors,
              edgecolor='black', linewidth=0.6)
best_i = list(ORDER).index(best_hpo)
bars[best_i].set_color(PALETTE[1]); bars[best_i].set_edgecolor('black')
ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER, rotation=20, ha='right')
ax.set_ylabel('Accuracy (10-fold CV)')
ax.set_ylim(accs.min()-0.008, accs.max()+0.008)
ax.set_title('Accuracy of 7 LightGBM Configurations (Baseline + 6 HPO)')
for b, v in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, v+0.0012, f'{v:.4f}', ha='center', fontsize=8.5)
plt.tight_layout(); plt.savefig(f'{FIG}/fig07_hpo_accuracy.png'); plt.show()

In [22]:
# --- Figure: boxplot + convergence ---
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
bx = ax[0].boxplot([hpo_folds[n]['accuracy'].values for n in ORDER], labels=ORDER,
                   patch_artist=True, widths=0.55, medianprops=dict(color='black'))
for patch, c in zip(bx['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax[0].set_ylabel('Accuracy'); ax[0].set_title('10-Fold Accuracy per Configuration')
ax[0].tick_params(axis='x', rotation=25)
for n in ORDER[1:]:
    h = hpo_history[n]
    ax[1].plot(range(1, len(h)+1), h, marker='o', ms=3, label=n)
ax[1].set_xlabel('Trial'); ax[1].set_ylabel('Best CV Accuracy (cumulative)')
ax[1].set_title('Hyperparameter Search Convergence'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig08_hpo_boxplot_convergence.png'); plt.show()

In [23]:
# --- Friedman + Nemenyi ---
acc_matrix = {n: hpo_folds[n]['accuracy'].tolist() for n in ORDER}
chi2, p_fried = stats.friedmanchisquare(*[acc_matrix[n] for n in ORDER])
ranks = pd.DataFrame(acc_matrix).rank(axis=1, ascending=False)
mean_ranks = ranks.mean()
k, N = len(ORDER), 10
q_alpha = 2.949  # studentized range, k=7, alpha=0.05
CD = q_alpha * np.sqrt(k*(k+1)/(6.0*N))
print(f'Friedman: chi2={chi2:.4f}, p={p_fried:.4f}, df={k-1}')
print(f'Critical Difference (Nemenyi, alpha=0.05) = {CD:.4f}')
print('\nMean rank (lower is better):')
print(mean_ranks.sort_values().round(3).to_string())

nem = sp.posthoc_nemenyi_friedman(np.array([acc_matrix[n] for n in ORDER]).T)
nem.index = ORDER; nem.columns = ORDER
json.dump({'chi2':float(chi2),'p':float(p_fried),'CD':float(CD),
           'mean_ranks':{n:float(mean_ranks[n]) for n in ORDER}},
          open(f'{OUT}/stage1_friedman.json','w'), indent=2)

Friedman: chi2=9.1271, p=0.1666, df=6
Critical Difference (Nemenyi, alpha=0.05) = 2.8490

Mean rank (lower is better):
Random Search         2.95
scikit-optimize GP    3.40
Halving               3.75
Optuna TPE            3.75
Hyperopt TPE          3.90
Baseline              5.10
Grid Search           5.15


If the Friedman test is not significant (p > 0.05), the Critical Difference diagram and Nemenyi matrix below are shown **descriptively**, to visualise the mean ranks.

In [24]:
# --- Figure: Critical Difference diagram + Nemenyi heatmap ---
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
# CD diagram
try:
    sp.critical_difference_diagram(mean_ranks, nem, ax=ax[0],
        label_props={'fontsize':9}, crossbar_props={'linewidth':2})
    ax[0].set_title(f'Critical Difference Diagram (CD={CD:.3f}, α=0.05)')
except Exception as e:
    mr = mean_ranks.sort_values()
    ax[0].barh(range(len(mr)), mr.values, color=PALETTE[0], edgecolor='black')
    ax[0].set_yticks(range(len(mr))); ax[0].set_yticklabels(mr.index)
    ax[0].invert_yaxis(); ax[0].set_xlabel('Mean Rank')
    ax[0].set_title(f'Mean Rank Friedman (CD={CD:.3f})')
sns.heatmap(nem, annot=True, fmt='.3f', cmap='RdYlGn', center=0.05, vmin=0, vmax=1,
            square=True, linewidths=0.5, ax=ax[1], cbar_kws={'label':'Nemenyi p-value'})
ax[1].set_title('Nemenyi Post-Hoc p-value')
plt.tight_layout(); plt.savefig(f'{FIG}/fig09_critical_difference.png'); plt.show()

In [25]:
# --- Figure: efficiency (time vs accuracy) ---
fig, ax = plt.subplots(figsize=(9, 5.5))
for i, n in enumerate(ORDER[1:], 1):
    ax.scatter(hpo_time[n], hpo_summary.loc[n,'accuracy'], s=140, color=PALETTE[(i-1)%5],
               edgecolor='black', zorder=3)
    ax.annotate(n, (hpo_time[n], hpo_summary.loc[n,'accuracy']),
                xytext=(6,4), textcoords='offset points', fontsize=9)
ax.set_xlabel('Optimization Time (s)'); ax.set_ylabel('10-fold CV Accuracy')
ax.set_title('HPO Efficiency: Time vs Accuracy')
plt.tight_layout(); plt.savefig(f'{FIG}/fig10_hpo_efficiency.png'); plt.show()

## 6. Final Model Training

The configuration with the highest mean 10-fold CV accuracy is used to train the final model on the full training set. It is then evaluated on the test set (4,815 samples), which was never used during optimization.

In [26]:
best_params_final = hpo_params[best_hpo]
print(f'Model final: LightGBM + {best_hpo}')
print('Selected hyperparameters:')
for kk, vv in best_params_final.items():
    print(f'  {kk}: {vv}')
final_model = LGBMClassifier(**{**best_params_final, **DEFAULT})
final_model.fit(X_train, y_train)

yp = final_model.predict(X_test); pp = final_model.predict_proba(X_test)[:, 1]
final_metrics = dict(
    accuracy=accuracy_score(y_test, yp),
    precision=precision_score(y_test, yp, average='weighted'),
    recall=recall_score(y_test, yp, average='weighted'),
    f1=f1_score(y_test, yp, average='weighted'),
    mcc=matthews_corrcoef(y_test, yp),
    auc=roc_auc_score(y_test, pp))
print('\nFinal model holdout evaluation:')
for kk, vv in final_metrics.items():
    print(f'  {kk:10s}= {vv:.4f}')
json.dump({k: float(v) for k,v in final_metrics.items()}, open(f'{OUT}/final_metrics.json','w'), indent=2)

Model final: LightGBM + Random Search
Selected hyperparameters:
  colsample_bytree: 0.5324461235544908
  learning_rate: 0.01275726949987024
  max_depth: 9
  min_child_samples: 6
  n_estimators: 486
  num_leaves: 127
  reg_alpha: 2.1517032190107035e-07
  reg_lambda: 9.542548269555452
  subsample: 0.6333905071376424



Final model holdout evaluation:
  accuracy  = 0.8201
  precision = 0.8319
  recall    = 0.8201
  f1        = 0.8218
  mcc       = 0.6433
  auc       = 0.9043


## 7. Stage 2 — Model Interpretation with SHAP

TreeSHAP (default `feature_perturbation='tree_path_dependent'`, no separate background data) is computed on the test set. Values are on the *log-odds* scale for the ADHERENT class. Four levels of analysis:

- **L1** — global feature importance (bar and beeswarm)
- **L2** — feature interactions (dependence plots)
- **L3** — mean direction of contribution toward ADHERENT (signed mean SHAP)
- **L4** — individual patient explanation (waterfall plot)

SHAP values are **associative** with the model output, not causal effects on adherence.

In [27]:
# --- SHAP values (TreeSHAP) ---
explainer = shap.TreeExplainer(final_model)
sv = explainer.shap_values(X_test)
if isinstance(sv, list):
    sv = sv[1]              # positive class (ADHERENT)
print('SHAP values shape:', np.array(sv).shape)
mean_abs = np.abs(sv).mean(0)
shap_rank = pd.DataFrame({'feature': X_test.columns, 'mean_abs_shap': mean_abs}
                         ).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
print('\nRanking Global Feature Importance (mean |SHAP|):')
print(shap_rank.to_string())
shap_rank.to_csv(f'{OUT}/shap_global_importance.csv', index=False)

SHAP values shape: (4815, 17)

Ranking Global Feature Importance (mean |SHAP|):
                                    feature  mean_abs_shap
0                                UNITSTOTAL       1.304842
1                                 LOG_UNITS       0.507247
2                         ANNUALCLAIMAMOUNT       0.332723
3                             UNITS_PER_AGE       0.156481
4   COMPLICATIONDEVELOPMENT_NO_COMPLICATION       0.135381
5                                       AGE       0.127260
6                                 LOG_CLAIM       0.120431
7                            CLAIM_PER_UNIT       0.117455
8                    DIAGNOSIS_HYPERTENSION       0.066960
9                               CLAIM_RATIO       0.061268
10                                 GENDER_M       0.023252
11                       ANNUALCONTRIBUTION       0.022886
12               COMORBIDITY_NO_COMORBIDITY       0.012137
13                              LOG_CONTRIB       0.010057
14                       COVERTYPE_

In [28]:
# --- L1a: global SHAP bar plot ---
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_test, plot_type='bar', show=False, max_display=17)
plt.title('SHAP Global Feature Importance (mean |SHAP|)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG}/fig11_shap_bar.png'); plt.show()

In [29]:
# --- L1b: SHAP beeswarm ---
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_test, show=False, max_display=17)
plt.title('SHAP Beeswarm Plot', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG}/fig12_shap_beeswarm.png'); plt.show()

In [30]:
# --- L2: dependence plots of the top 4 features ---
top4 = shap_rank['feature'].head(4).tolist()
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for i, feat in enumerate(top4):
    r, c = divmod(i, 2)
    shap.dependence_plot(feat, sv, X_test, ax=axes[r, c], show=False,
                         interaction_index='auto')
    axes[r, c].set_title(f'Dependence: {feat}', fontsize=11)
plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG}/fig13_shap_dependence.png'); plt.show()

In [31]:
# --- L3: mean direction of contribution (signed mean SHAP) ---
sv_df = pd.DataFrame(sv, columns=X_test.columns)
mean_signed = sv_df.mean(0).sort_values()
plt.figure(figsize=(9, 6))
colors_bar = [PALETTE[1] if v < 0 else PALETTE[2] for v in mean_signed.values]
plt.barh(range(len(mean_signed)), mean_signed.values, color=colors_bar, edgecolor='black')
plt.yticks(range(len(mean_signed)), mean_signed.index)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Mean SHAP value (direction toward ADHERENT)')
plt.title('Signed Mean SHAP: Direction of Contribution per Feature', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG}/fig14_shap_direction.png'); plt.show()

In [32]:
# --- L4: individual waterfall (patient #0) ---
expl = shap.Explanation(values=sv, base_values=explainer.expected_value[1]
                        if isinstance(explainer.expected_value, (list, np.ndarray))
                        else explainer.expected_value,
                        data=X_test.values, feature_names=list(X_test.columns))
fig = plt.figure(figsize=(10, 5))
shap.plots.waterfall(expl[0], max_display=12, show=False)
plt.title(f'SHAP Waterfall — Patient #0 (actual={y_test.iloc[0]}, predicted={yp[0]})',
          fontsize=11, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG}/fig15_shap_waterfall_p0.png'); plt.show()

### 7.1 SHAP direction statistics (basis for the discussion in thesis section 4.3)

In [33]:
from scipy.stats import spearmanr
sv_df = pd.DataFrame(sv, columns=X_test.columns)
rows = []
for c in X_test.columns:
    x = X_test[c].values
    if set(np.unique(x)) <= {0, 1}:
        direction = (f"value=1: {sv_df[c][x == 1].mean():+.3f} (n={int(x.sum())}) | "
                     f"value=0: {sv_df[c][x == 0].mean():+.3f} (n={int((x == 0).sum())})")
    else:
        direction = f"Spearman(feature value, SHAP) = {spearmanr(x, sv_df[c]).correlation:+.2f}"
    rows.append(dict(feature=c, mean_abs=np.abs(sv_df[c]).mean(), signed_mean=sv_df[c].mean(), direction=direction))
shap_dir = pd.DataFrame(rows).sort_values('mean_abs', ascending=False).reset_index(drop=True)
shap_dir.to_csv(f'{OUT}/shap_direction.csv', index=False)
pd.set_option('display.max_colwidth', 120)
print(shap_dir.round(4).to_string(index=False))

u, s_u = X_test['UNITSTOTAL'].values, sv_df['UNITSTOTAL'].values
print(f"\nMean SHAP of UNITSTOTAL for UNITSTOTAL <= 100 : {s_u[u <= 100].mean():+.3f}")

data_all = X.join(y)
print('\nShare of ADHERENT by complication status (full data, 0 = developed complications):')
print(data_all.groupby('COMPLICATIONDEVELOPMENT_NO_COMPLICATION')['ADHERENCE'].agg(['mean', 'count']).round(3))
print('\nShare of ADHERENT by age quartile:')
print(data_all.groupby(pd.qcut(data_all['AGE'], 4))['ADHERENCE'].mean().round(3))

                                feature  mean_abs  signed_mean                                           direction
                             UNITSTOTAL    1.3048       0.0493               Spearman(feature value, SHAP) = +0.89
                              LOG_UNITS    0.5072       0.0197               Spearman(feature value, SHAP) = +0.89
                      ANNUALCLAIMAMOUNT    0.3327       0.0078               Spearman(feature value, SHAP) = +0.96
                          UNITS_PER_AGE    0.1565      -0.0021               Spearman(feature value, SHAP) = +0.82
COMPLICATIONDEVELOPMENT_NO_COMPLICATION    0.1354       0.0151  value=1: +0.083 (n=4361) | value=0: -0.638 (n=454)
                                    AGE    0.1273      -0.0074               Spearman(feature value, SHAP) = +0.83
                              LOG_CLAIM    0.1204       0.0040               Spearman(feature value, SHAP) = +0.96
                         CLAIM_PER_UNIT    0.1175      -0.0235               Spe

## 8. Final Model Evaluation

Metrics use the **default 0.5 threshold**. Overall columns are weighted averages; per-class metrics are added because the clinical goal is to detect **NON-ADHERENT** patients. The Youden threshold is computed from the test-set ROC curve and is **exploratory only**; it is not used for the final metrics.

In [34]:
# --- Confusion matrix + ROC model final ---
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_test, yp)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax[0],
            xticklabels=['NON-ADHERENT','ADHERENT'], yticklabels=['NON-ADHERENT','ADHERENT'])
ax[0].set_title(f'Confusion Matrix — LightGBM Final\n(accuracy={final_metrics["accuracy"]:.4f})')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
fpr, tpr, thr = roc_curve(y_test, pp)
j = tpr - fpr; jstar = thr[np.argmax(j)]
ax[1].plot(fpr, tpr, color=PALETTE[0], lw=2.5, label=f'AUC = {final_metrics["auc"]:.4f}')
ax[1].plot([0,1],[0,1],'k--', lw=1)
ax[1].scatter(fpr[np.argmax(j)], tpr[np.argmax(j)], color=PALETTE[1], s=90, zorder=3,
              label=f"Youden's J* (thr={jstar:.3f})")
ax[1].set_xlabel('False Positive Rate'); ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curve — Final LightGBM'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig(f'{FIG}/fig16_final_cm_roc.png'); plt.show()
print(f"Youden's J* threshold = {jstar:.4f} | sensitivity={tpr[np.argmax(j)]:.4f} specificity={1-fpr[np.argmax(j)]:.4f}")

Youden's J* threshold = 0.3831 | sensitivity=0.9323 specificity=0.7313


In [35]:
print(classification_report(y_test, yp, target_names=['NON-ADHERENT', 'ADHERENT'], digits=4))
tn, fp_, fn, tp = confusion_matrix(y_test, yp).ravel()
print(f'TN={tn}  FP={fp_}  FN={fn}  TP={tp}')
print(f'Recall NON-ADHERENT = {tn/(tn+fp_):.4f}  |  Recall ADHERENT = {tp/(tp+fn):.4f}')
jbest = np.argmax(tpr - fpr)
print(f"\nYouden (exploratory): threshold={thr[jbest]:.3f}, J={tpr[jbest]-fpr[jbest]:.3f}, "
      f"sensitivity={tpr[jbest]:.3f}, specificity={1-fpr[jbest]:.3f}")

              precision    recall  f1-score   support

NON-ADHERENT     0.8974    0.7897    0.8401      2881
    ADHERENT     0.7342    0.8656    0.7945      1934

    accuracy                         0.8201      4815
   macro avg     0.8158    0.8276    0.8173      4815
weighted avg     0.8319    0.8201    0.8218      4815

TN=2275  FP=606  FN=260  TP=1674
Recall NON-ADHERENT = 0.7897  |  Recall ADHERENT = 0.8656

Youden (exploratory): threshold=0.383, J=0.664, sensitivity=0.932, specificity=0.731


## 9. Sensitivity Analysis

Three robustness checks, using the same split, seed and final hyperparameters:

1. **Class weighting**: final model with vs without `class_weight='balanced'`.
2. **Claim-unit feature ablation**: the ADHERENCE label and `UNITSTOTAL` are derived from the same claims data, so the model's reliance on unit-based features is measured.
3. **Baseline vs final on the test set**: paired bootstrap 95% CI (B = 2,000) and exact McNemar test.

In [36]:
from scipy.stats import binomtest
INT_PARAMS = {'n_estimators', 'max_depth', 'num_leaves', 'min_child_samples'}
BEST = {k: (int(v) if k in INT_PARAMS else v) for k, v in best_params_final.items()}

def full_metrics(y_true, y_pred, proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(accuracy=accuracy_score(y_true, y_pred),
                precision_w=precision_score(y_true, y_pred, average='weighted'),
                f1_w=f1_score(y_true, y_pred, average='weighted'),
                mcc=matthews_corrcoef(y_true, y_pred), auc=roc_auc_score(y_true, proba),
                recall_non_adherent=tn / (tn + fp), recall_adherent=tp / (tp + fn),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def fit_eval(params, cols=None, extra=None):
    cols = cols or list(X_train.columns)
    m = LGBMClassifier(**{**params, **DEFAULT, **(extra or {})}).fit(X_train[cols], y_train)
    pred, proba = m.predict(X_test[cols]), m.predict_proba(X_test[cols])[:, 1]
    return full_metrics(y_test, pred, proba), pred, proba

def cv10_short(params, extra=None):
    acc, mcc = [], []
    for tr, te in CV10.split(X_train, y_train):
        m = LGBMClassifier(**{**params, **DEFAULT, **(extra or {})}).fit(X_train.iloc[tr], y_train.iloc[tr])
        p = m.predict(X_train.iloc[te])
        acc.append(accuracy_score(y_train.iloc[te], p)); mcc.append(matthews_corrcoef(y_train.iloc[te], p))
    return dict(acc_mean=float(np.mean(acc)), acc_std=float(np.std(acc)),
                mcc_mean=float(np.mean(mcc)), mcc_std=float(np.std(mcc)))

sens = {}
final_m, final_pred, final_proba = fit_eval(BEST)
assert abs(final_m['accuracy'] - final_metrics['accuracy']) < 1e-12, 'Final model not reproduced'

### 9.1 Class weighting (thesis Table 4.5)

In [37]:
cw_m, _, _ = fit_eval(BEST, extra={'class_weight': 'balanced'})
sens['class_weight'] = dict(unweighted=dict(holdout=final_m, cv10=cv10_short(BEST)),
                            balanced=dict(holdout=cw_m, cv10=cv10_short(BEST, extra={'class_weight': 'balanced'})))
tab45 = pd.DataFrame({'Unweighted (final)': final_m, "class_weight='balanced'": cw_m}).T
print(tab45[['accuracy', 'mcc', 'auc', 'recall_non_adherent', 'recall_adherent']].round(4).to_string())
print('\n10-fold CV:', {k: {kk: round(vv, 4) for kk, vv in v['cv10'].items()} for k, v in sens['class_weight'].items()})

                         accuracy     mcc     auc  recall_non_adherent  recall_adherent
Unweighted (final)         0.8201  0.6433  0.9043               0.7897           0.8656
class_weight='balanced'    0.8160  0.6539  0.9046               0.7459           0.9204

10-fold CV: {'unweighted': {'acc_mean': 0.8241, 'acc_std': 0.0085, 'mcc_mean': 0.6477, 'mcc_std': 0.0168}, 'balanced': {'acc_mean': 0.8189, 'acc_std': 0.0087, 'mcc_mean': 0.6546, 'mcc_std': 0.0157}}


### 9.2 Claim-unit feature ablation (thesis Table 4.6)

In [38]:
all_cols = list(X_train.columns)
scenarios = {
    'Final model (17 features)': [],
    'Without UNITSTOTAL, LOG_UNITS, UNITS_PER_AGE': ['UNITSTOTAL', 'LOG_UNITS', 'UNITS_PER_AGE'],
    'Without all unit-derived features (+ CLAIM_PER_UNIT)': ['UNITSTOTAL', 'LOG_UNITS', 'UNITS_PER_AGE', 'CLAIM_PER_UNIT'],
    'UNITSTOTAL only': [c for c in all_cols if c != 'UNITSTOTAL'],
}
abl = {}
for name, drop in scenarios.items():
    cols = [c for c in all_cols if c not in drop]
    m_, _, _ = fit_eval(BEST, cols=cols)
    abl[name] = dict(n_features=len(cols), features=cols, holdout=m_)
sens['ablation'] = abl
tab46 = pd.DataFrame({k: {'n_features': v['n_features'], **v['holdout']} for k, v in abl.items()}).T
print(tab46[['n_features', 'accuracy', 'mcc', 'auc', 'recall_non_adherent']].to_string())

                                                      n_features  accuracy       mcc       auc  recall_non_adherent
Final model (17 features)                                   17.0  0.820145  0.643325  0.904295             0.789656
Without UNITSTOTAL, LOG_UNITS, UNITS_PER_AGE                14.0  0.815161  0.634127  0.898915             0.783061
Without all unit-derived features (+ CLAIM_PER_UNIT)        13.0  0.758048  0.514564  0.838693             0.742451
UNITSTOTAL only                                              1.0  0.796885  0.603346  0.873473             0.752864


### 9.3 Baseline vs final model on the test set (thesis Table 4.7)

In [39]:
base_m, base_pred, base_proba = fit_eval({})
rng = np.random.default_rng(RANDOM_STATE)
yt, B = y_test.to_numpy(), 2000
boot = {k: [] for k in ('final_acc', 'final_mcc', 'final_auc', 'd_acc', 'd_mcc', 'd_auc')}
for _ in range(B):
    idx = rng.integers(0, len(yt), len(yt))
    if yt[idx].min() == yt[idx].max():
        continue
    fa, ba = accuracy_score(yt[idx], final_pred[idx]), accuracy_score(yt[idx], base_pred[idx])
    fm, bm = matthews_corrcoef(yt[idx], final_pred[idx]), matthews_corrcoef(yt[idx], base_pred[idx])
    fu, bu = roc_auc_score(yt[idx], final_proba[idx]), roc_auc_score(yt[idx], base_proba[idx])
    for k, v in zip(boot, (fa, fm, fu, fa - ba, fm - bm, fu - bu)):
        boot[k].append(v)
ci = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in boot.items()}
f_ok, b_ok = final_pred == yt, base_pred == yt
n01, n10 = int(np.sum(f_ok & ~b_ok)), int(np.sum(~f_ok & b_ok))
p_mcnemar = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
sens['baseline_vs_final'] = dict(baseline=base_m, final=final_m, bootstrap_B=B, ci95=ci,
                                 mcnemar=dict(final_right_baseline_wrong=n01, final_wrong_baseline_right=n10,
                                              p_value=float(p_mcnemar)))
tab47 = pd.DataFrame({'Baseline (default)': base_m, 'Final (Random Search)': final_m}).T
tab47.loc['Difference (final - baseline)'] = tab47.iloc[1] - tab47.iloc[0]
print(tab47[['accuracy', 'f1_w', 'mcc', 'auc', 'recall_non_adherent']].round(4).to_string())
print(f"\n95% CI final model : acc {np.round(ci['final_acc'], 3)} | MCC {np.round(ci['final_mcc'], 3)} | AUC {np.round(ci['final_auc'], 3)}")
print(f"95% CI difference  : acc {np.round(ci['d_acc'], 4)} | MCC {np.round(ci['d_mcc'], 4)} | AUC {np.round(ci['d_auc'], 4)}")
print(f'Exact McNemar      : final right/baseline wrong = {n01}, the reverse = {n10}, p = {p_mcnemar:.4f}')
json.dump(sens, open(f'{OUT}/sensitivity_analysis.json', 'w', encoding='utf-8'), indent=2, default=float)

                               accuracy    f1_w     mcc     auc  recall_non_adherent
Baseline (default)               0.8199  0.8216  0.6432  0.9036               0.7886
Final (Random Search)            0.8201  0.8218  0.6433  0.9043               0.7897
Difference (final - baseline)    0.0002  0.0002  0.0001  0.0007               0.0010

95% CI final model : acc [0.81  0.831] | MCC [0.623 0.664] | AUC [0.896 0.913]
95% CI difference  : acc [-0.0044  0.0048] | MCC [-0.0091  0.0095] | AUC [-0.0011  0.0025]
Exact McNemar      : final right/baseline wrong = 67, the reverse = 66, p = 1.0000


## 10. Experiment Summary

In [40]:
fr = json.load(open(f'{OUT}/stage1_friedman.json'))
print('=' * 72)
print('EXPERIMENT SUMMARY')
print('=' * 72)
print(f'\n[Preliminary study] Highest mean CV accuracy among boosting algorithms: {best_boost}')
print(boost_summary[['accuracy', 'auc', 'mcc']].to_string())
print(f'\n[Stage 1] Selected HPO configuration (highest mean 10-fold CV accuracy): {best_hpo}')
print(hpo_summary[['accuracy', 'mcc', 'auc', 'time_s']].to_string())
print(f"Friedman chi2 = {fr['chi2']:.3f}, p = {fr['p']:.3f}  ->  differences between methods are "
      f"{'significant' if fr['p'] < 0.05 else 'not significant'} (alpha = 0.05)")
print('\n[Stage 2] SHAP: top three features by mean |SHAP|')
print(shap_rank.head(3).to_string(index=False))
print('\n[Evaluation] Final model on the test set (threshold 0.5)')
for kk, vv in final_metrics.items():
    print(f'   {kk:10s} = {vv:.4f}')
print(f"   recall NON-ADHERENT = {final_m['recall_non_adherent']:.4f}")
print(f"\n[Sensitivity] Final vs baseline: accuracy difference = {final_m['accuracy'] - base_m['accuracy']:+.4f}, "
      f'McNemar p = {p_mcnemar:.3f}')
print('\nFigures saved in figures/, numerical results in outputs/.')

EXPERIMENT SUMMARY

[Preliminary study] Highest mean CV accuracy among boosting algorithms: XGBoost
                   accuracy     auc     mcc
XGBoost              0.8184  0.9029  0.6551
AdaBoost             0.7873  0.8725  0.6222
Gradient Boosting    0.8116  0.8992  0.6492
LightGBM             0.8173  0.9028  0.6508
CatBoost             0.8123  0.9005  0.6492

[Stage 1] Selected HPO configuration (highest mean 10-fold CV accuracy): Random Search
                    accuracy     mcc     auc     time_s
Baseline              0.8218  0.6430  0.9025     0.0000
Grid Search           0.8211  0.6423  0.9033   632.3898
Random Search         0.8241  0.6477  0.9046   185.2108
Halving               0.8223  0.6434  0.9026   284.9333
Optuna TPE            0.8234  0.6464  0.9041   455.1507
Hyperopt TPE          0.8226  0.6455  0.9035   747.4926
scikit-optimize GP    0.8234  0.6473  0.9046  4003.8932
Friedman chi2 = 9.127, p = 0.167  ->  differences between methods are not significant (alpha = 0.05)